In [ ]:
import os
import glob
import shutil

BASE_SAVE_PATH = os.getcwd()

os.makedirs(os.path.join(BASE_SAVE_PATH, "geometry"), exist_ok=True)
os.makedirs(os.path.join(BASE_SAVE_PATH, "chemistry"), exist_ok=True)
os.makedirs(os.path.join(BASE_SAVE_PATH, "topography"), exist_ok=True)
os.makedirs(os.path.join(BASE_SAVE_PATH, "dataset"), exist_ok=True)
os.makedirs(os.path.join(BASE_SAVE_PATH, "models"), exist_ok=True)
os.makedirs(os.path.join(BASE_SAVE_PATH, "figures"), exist_ok=True)
print(f"Working directory tree created under {BASE_SAVE_PATH}")

SITE_DENSITY_MOL_CM2 = 1.66e-9
M_AL2O3_G_MOL = 101.96
RHO_AL2O3_G_CM3 = 3.1


# Al2O3 ALD on a 42 nm fin


## Libraries


In [ ]:
import importlib, sys, os, subprocess, ctypes

CUDA_LIBS = [
    "libnvJitLink.so.12",
    "libcublasLt.so.12",
    "libcublas.so.12",
    "libcusparse.so.12",
    "libcusolver.so.12",
    "libcudart.so.12",
]

for libname in CUDA_LIBS:
    try:
        print(f"Locating {libname} on filesystem...")
        find_cmd = f"find /usr/local /opt -name '{libname}' 2>/dev/null"
        paths = subprocess.check_output(find_cmd, shell=True, text=True).strip().split('\n')
        paths = [p for p in paths if p]

        if paths:
            lib_path = paths[0]
            print(f"[FIX] Found {libname} at: {lib_path}. Loading globally...")
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        else:
            print(f"[FIX] Warning: {libname} not found. Imports may fail.")
    except Exception as e:
        print(f"[FIX] File search error for {libname}: {e}")

REQUIRED = ["cantera", "viennaps", "pyvista", "physicsnemo", "torch"]
for _mod in REQUIRED:
    importlib.import_module(_mod)
    print(f"[OK] {_mod} imported successfully "
          f"(version: {getattr(sys.modules[_mod], '__version__', 'n/a')})")

BASE_SAVE_PATH = globals().get("BASE_SAVE_PATH", os.getcwd())

os.makedirs(f"{BASE_SAVE_PATH}", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/geometry", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/chemistry", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/topography", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/dataset", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/models", exist_ok=True)
os.makedirs(f"{BASE_SAVE_PATH}/figures", exist_ok=True)
print(f"Working directory tree created under {BASE_SAVE_PATH}/")


## Fin geometry


In [ ]:
import numpy as np
import pyvista as pv
import os

def _ensure_headless_rendering():
    try:
        _probe = pv.Plotter(off_screen=True, window_size=(64, 64))
        _probe.add_mesh(pv.Sphere())
        _probe.screenshot("/tmp/_pv_headless_probe.png")
        _probe.close()
        print("PyVista off-screen rendering verified (no Xvfb required).")
    except Exception as e:
        print(f"Off-screen render probe failed ({e}); falling back to Xvfb.")
        import subprocess
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "xvfb", "libgl1-mesa-glx"],
            check=False,
        )
        try:
            from pyvistaqt import BackgroundPlotter
        except ImportError:
            pass
        import os
        os.environ["DISPLAY"] = ":99"
        subprocess.Popen(
            ["Xvfb", ":99", "-screen", "0", "1280x1024x24"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        import time
        time.sleep(3)
        print("Started Xvfb virtual framebuffer on DISPLAY=:99.")

_ensure_headless_rendering()

FIN_HEIGHT_NM   = 42.0
FIN_WIDTH_NM    = 8.0
FIN_WIDTH_TOP_NM = 6.0
FIN_PITCH_NM    = 42.0
N_FINS          = 5
FIN_LENGTH_NM   = 120.0
STI_THICKNESS_NM = 0.0
SUBSTRATE_DEPTH_NM = 20.0
DOMAIN_MARGIN_NM = 20.0

domain_x = N_FINS * FIN_PITCH_NM + 2 * DOMAIN_MARGIN_NM
domain_y = FIN_LENGTH_NM + 2 * DOMAIN_MARGIN_NM

def make_tapered_fin(x_center, y0, y1, z0, z1, w_base, w_top):
    hb, ht = w_base / 2.0, w_top / 2.0
    base_pts = np.array([
        [x_center - hb, y0, z0], [x_center + hb, y0, z0],
        [x_center + hb, y1, z0], [x_center - hb, y1, z0],
    ])
    top_pts = np.array([
        [x_center - ht, y0, z1], [x_center + ht, y0, z1],
        [x_center + ht, y1, z1], [x_center - ht, y1, z1],
    ])
    pts = np.vstack([base_pts, top_pts])
    faces = np.hstack([
        [4, 0, 1, 2, 3],
        [4, 4, 5, 6, 7],
        [4, 0, 1, 5, 4],
        [4, 1, 2, 6, 5],
        [4, 2, 3, 7, 6],
        [4, 3, 0, 4, 7],
    ])
    fin = pv.PolyData(pts, faces)
    return fin.triangulate().clean()

def make_box(xmin, xmax, ymin, ymax, zmin, zmax):
    return pv.Box(bounds=(xmin, xmax, ymin, ymax, zmin, zmax))

vtp_path = f"{BASE_SAVE_PATH}/geometry/post_photolith_finfet.vtp"

if os.path.exists(vtp_path):
    print(f"Checkpoint found. Loading geometry from {vtp_path}")
    finfet_surface = pv.read(vtp_path)
else:

    substrate = make_box(
        -domain_x / 2, domain_x / 2,
        -domain_y / 2, domain_y / 2,
        -SUBSTRATE_DEPTH_NM, 0.0,
    )
    substrate["MaterialIds"] = np.full(substrate.n_points, 1)

    sti_field = make_box(
        -domain_x / 2, domain_x / 2,
        -domain_y / 2, domain_y / 2,
        -0.001, 0.0,
    )
    sti_field["MaterialIds"] = np.full(sti_field.n_points, 2)

    fin_blocks = []
    first_fin_x = -((N_FINS - 1) * FIN_PITCH_NM) / 2.0
    for i in range(N_FINS):
        xc = first_fin_x + i * FIN_PITCH_NM
        fin = make_tapered_fin(
            xc,
            -FIN_LENGTH_NM / 2, FIN_LENGTH_NM / 2,
            0.0, FIN_HEIGHT_NM,
            FIN_WIDTH_NM, FIN_WIDTH_TOP_NM,
        )
        fin["MaterialIds"] = np.full(fin.n_points, 1)
        fin_blocks.append(fin)

    fin_array = fin_blocks[0]
    for f in fin_blocks[1:]:
        fin_array = fin_array.merge(f)

    finfet_surface = substrate.extract_surface().merge(sti_field).merge(fin_array)
    finfet_surface = finfet_surface.triangulate().clean()

    finfet_surface.save(vtp_path)
    print(f"Saved post-photolithography FinFET geometry to: {vtp_path}")
    print(f"  Points : {finfet_surface.n_points}")
    print(f"  Cells  : {finfet_surface.n_cells}")
    print(f"  Bounds (nm): {finfet_surface.bounds}")
    print(f"  Fins   : {N_FINS}  (height={FIN_HEIGHT_NM} nm, base width={FIN_WIDTH_NM} nm, "
          f"top width={FIN_WIDTH_TOP_NM} nm, pitch={FIN_PITCH_NM} nm)")

plotter = pv.Plotter(off_screen=True, window_size=(1000, 700))
plotter.add_mesh(finfet_surface, scalars="MaterialIds", cmap=["#c9c9c9", "#4a7fb5"],
                  show_scalar_bar=False)
plotter.camera_position = "iso"
plotter.add_axes()
plotter.screenshot(f"{BASE_SAVE_PATH}/figures/post_photolith_finfet_preview.png")
print(f"Saved geometry preview screenshot to {BASE_SAVE_PATH}/figures/post_photolith_finfet_preview.png")


## TMA and water surface mechanism


In [ ]:
import cantera as ct
import numpy as np
import pandas as pd

MECHANISM_YAML = f"{BASE_SAVE_PATH}/chemistry/ald_al2o3_mechanism.yaml"

_mechanism_text = r"""
generator: ald-al2o3-mechanism-builder
description: |-
  Surface reaction mechanism for thermal ALD of Al2O3 from
  trimethylaluminum (TMA, Al(CH3)3) and H2O.
cantera-version: 3.2.0

units: {length: cm, time: s, quantity: mol, activation-energy: kJ/mol}

phases:
  - name: gas
    thermo: ideal-gas
    elements: [Al, C, H, O, N]
    species: [TMA, H2O, CH4, N2]
    state: {T: 300.0, P: 1.0 atm}

  - name: Al2O3_bulk
    thermo: fixed-stoichiometry
    elements: [Al, O]
    species: [Al2O3s]

  - name: Al2O3_surface
    thermo: ideal-surface
    elements: [Al, C, H, O, N]
    species: [OH_s, O_Al_CH3_2_s]
    kinetics: surface
    reactions: all
    site-density: 1.66e-9   # mol/cm^2 (~1e15 sites/cm^2, -OH density on hydroxylated oxide)
    adjacent-phases: [gas, Al2O3_bulk]
    state: {T: 300.0, coverages: {OH_s: 1.0, O_Al_CH3_2_s: 0.0}}

species:
  - name: TMA
    composition: {Al: 1, C: 3, H: 9}
    thermo:
      model: NASA7
      temperature-ranges: [200.0, 1000.0, 3500.0]
      data:
        - [4.0, 0.02, -1.0e-5, 2.0e-9, -1.5e-13, -8000.0, 10.0]
        - [4.0, 0.02, -1.0e-5, 2.0e-9, -1.5e-13, -8000.0, 10.0]

  - name: H2O
    composition: {H: 2, O: 1}
    thermo:
      model: NASA7
      temperature-ranges: [200.0, 1000.0, 3500.0]
      data:
        - [4.198, -2.036e-3, 6.520e-6, -5.487e-9, 1.771e-12, -30293.7, -0.849]
        - [3.033, 2.176e-3, -1.640e-7, -9.704e-11, 1.682e-14, -30004.3, 4.966]

  - name: CH4
    composition: {C: 1, H: 4}
    thermo:
      model: NASA7
      temperature-ranges: [200.0, 1000.0, 3500.0]
      data:
        - [5.149, -0.01367, 4.918e-5, -4.847e-8, 1.666e-11, -10246.6, -4.643]
        - [1.911, 9.603e-3, -3.567e-6, 6.075e-10, -3.923e-14, -10099.2, 8.483]

  - name: N2
    composition: {N: 2}
    thermo:
      model: NASA7
      temperature-ranges: [200.0, 1000.0, 3500.0]
      data:
        - [3.531, -1.237e-4, -5.030e-7, 2.435e-9, -1.409e-12, -1046.98, 2.968]
        - [2.953, 1.397e-3, -4.927e-7, 7.870e-11, -4.608e-15, -923.949, 5.872]

  - name: OH_s
    composition: {O: 1, H: 1}
    thermo:
      model: constant-cp
      T0: 300.0
      h0: -25.0 kJ/mol
      s0: 0.0 J/mol/K
      cp0: 0.0 J/mol/K

  - name: O_Al_CH3_2_s
    composition: {Al: 1, O: 1, C: 2, H: 6}
    thermo:
      model: constant-cp
      T0: 300.0
      h0: -120.0 kJ/mol
      s0: 0.0 J/mol/K
      cp0: 0.0 J/mol/K

  - name: Al2O3s
    composition: {Al: 2, O: 3}
    thermo:
      model: constant-cp
      T0: 300.0
      h0: -1675.7 kJ/mol
      s0: 50.9 J/mol/K
      cp0: 79.0 J/mol/K
    equation-of-state:
      model: constant-volume
      density: 3.1 g/cm^3

reactions:
  - equation: TMA + OH_s => O_Al_CH3_2_s + CH4
    rate-constant: {A: 3.6e+7, b: 0.5, Ea: 12.0}

  - equation: 2 O_Al_CH3_2_s + 3 H2O => 2 OH_s + Al2O3s + 4 CH4
    rate-constant: {A: 1.1e+8, b: 0.5, Ea: 18.0}
"""

with open(MECHANISM_YAML, "w") as f:
    f.write(_mechanism_text)
print(f"Wrote Cantera surface mechanism to: {MECHANISM_YAML}")

surf = ct.Interface(MECHANISM_YAML, "Al2O3_surface")
gas = surf.adjacent["gas"]
bulk = surf.adjacent["Al2O3_bulk"]

print("\nSurface species:", surf.species_names)
print("Gas species     :", gas.species_names)
print("Bulk species     :", bulk.species_names)
print("Reactions        :")
for r in surf.reactions():
    print("   ", r.equation)

TEMPERATURES_C = [150.0, 200.0, 250.0, 300.0]
TMA_MOLE_FRACTION = 1.0e-3
H2O_MOLE_FRACTION = 1.0e-3
PULSE_TIMES_S = np.logspace(-2, 0.5, 25)

saturation_records = []

for T_C in TEMPERATURES_C:
    T_K = T_C + 273.15
    for pulse_t in PULSE_TIMES_S:
        surf.TP = T_K, ct.one_atm
        surf.coverages = {"OH_s": 1.0, "O_Al_CH3_2_s": 0.0}
        gas.TPX = T_K, ct.one_atm, {"TMA": TMA_MOLE_FRACTION,
                                     "N2": 1.0 - TMA_MOLE_FRACTION}
        surf.advance_coverages(pulse_t)
        theta_tma_sat = surf["O_Al_CH3_2_s"].coverages[0]

        gas.TPX = T_K, ct.one_atm, {"H2O": H2O_MOLE_FRACTION,
                                     "N2": 1.0 - H2O_MOLE_FRACTION}
        surf.advance_coverages(pulse_t)
        theta_oh_regen = surf["OH_s"].coverages[0]

        saturation_records.append({
            "temperature_C": T_C,
            "temperature_K": T_K,
            "pulse_time_s": pulse_t,
            "theta_methyl_after_TMA_pulse": theta_tma_sat,
            "theta_hydroxyl_after_H2O_pulse": theta_oh_regen,
        })

saturation_df = pd.DataFrame(saturation_records)
saturation_csv = f"{BASE_SAVE_PATH}/chemistry/saturation_curves.csv"
saturation_df.to_csv(saturation_csv, index=False)
print(f"\nSaved {len(saturation_df)} Cantera-solved saturation data points to: {saturation_csv}")
print(saturation_df.head(10).to_string(index=False))

gpc_records = []
for T_C in TEMPERATURES_C:
    sub = saturation_df[saturation_df.temperature_C == T_C]
    theta_sat = sub["theta_methyl_after_TMA_pulse"].max()
    areal_mol_al2o3_cm2 = 0.5 * theta_sat * SITE_DENSITY_MOL_CM2
    areal_mass_g_cm2 = areal_mol_al2o3_cm2 * M_AL2O3_G_MOL
    gpc_cm = areal_mass_g_cm2 / RHO_AL2O3_G_CM3
    gpc_nm = gpc_cm * 1.0e7
    gpc_records.append({"temperature_C": T_C, "theta_saturated": theta_sat,
                         "GPC_nm_per_cycle": gpc_nm})

gpc_df = pd.DataFrame(gpc_records)
gpc_csv = f"{BASE_SAVE_PATH}/chemistry/growth_per_cycle.csv"
gpc_df.to_csv(gpc_csv, index=False)
print("\nCantera-derived growth-per-cycle (GPC) vs. temperature:")
print(gpc_df.to_string(index=False))
print(f"Saved to: {gpc_csv}")


## Level-set film growth


In [ ]:
import viennaps as vps
import numpy as np
import pandas as pd
import pyvista as pv
import os
import cantera as ct
from scipy.stats import qmc
from scipy.optimize import curve_fit

vps.setDimension(3)
vps.setNumThreads(4)

N_SAMPLES = 100
PARAM_BOUNDS = {
    "temperature_C": (150.0, 300.0),
    "pulse_time_s": (0.05, 2.0),
    "num_cycles": (4, 20),
    "tma_mole_fraction": (5e-4, 5e-3),
}
sampler = qmc.LatinHypercube(d=4, seed=42)
lhs_unit = sampler.random(n=N_SAMPLES)
bounds_lo = np.array([b[0] for b in PARAM_BOUNDS.values()])
bounds_hi = np.array([b[1] for b in PARAM_BOUNDS.values()])
lhs_scaled = qmc.scale(lhs_unit, bounds_lo, bounds_hi)

design_df = pd.DataFrame(lhs_scaled, columns=list(PARAM_BOUNDS.keys()))
design_df["num_cycles"] = design_df["num_cycles"].round().astype(int)

dataset_csv = f"{BASE_SAVE_PATH}/dataset/ald_high_fidelity_dataset.csv"
vtp_input_path = f"{BASE_SAVE_PATH}/geometry/post_photolith_finfet.vtp"

dataset_records = []
print(f"Executing rigorous 3D physics pipeline across {N_SAMPLES} automated conditions...")
print("This step generates the single source-of-truth for neural network training.")

for i, row in design_df.iterrows():

    surf_local = ct.Interface(MECHANISM_YAML, "Al2O3_surface")
    gas_local = surf_local.adjacent["gas"]
    T_K = row.temperature_C + 273.15

    times = np.logspace(-2, 0.5, 10)
    coverages = []
    for t in times:
        surf_local.TP = T_K, ct.one_atm
        surf_local.coverages = {"OH_s": 1.0, "O_Al_CH3_2_s": 0.0}
        gas_local.TPX = T_K, ct.one_atm, {"TMA": row.tma_mole_fraction, "N2": 1.0 - row.tma_mole_fraction}
        surf_local.advance_coverages(t)
        coverages.append(surf_local["O_Al_CH3_2_s"].coverages[0])

    def langmuir(t, theta_m, tau): return theta_m * (1.0 - np.exp(-t / tau))
    try:
        popt, _ = curve_fit(langmuir, times, coverages, p0=[1.0, 0.1], maxfev=2000)
        _, tau_tma_s = popt
    except:
        tau_tma_s = 0.1

    surf_local.TP = T_K, ct.one_atm
    surf_local.coverages = {"OH_s": 1.0, "O_Al_CH3_2_s": 0.0}
    gas_local.TPX = T_K, ct.one_atm, {"TMA": row.tma_mole_fraction, "N2": 1.0 - row.tma_mole_fraction}
    surf_local.advance_coverages(row.pulse_time_s)

    theta_sat = surf_local["O_Al_CH3_2_s"].coverages[0]
    areal_mol_al2o3_cm2 = 0.5 * theta_sat * SITE_DENSITY_MOL_CM2
    gpc_nm = (areal_mol_al2o3_cm2 * M_AL2O3_G_MOL / RHO_AL2O3_G_CM3) * 1.0e7

    local_domain = vps.Domain()
    try:
        vps.ImportVTP(domain=local_domain, filename=vtp_input_path).apply()
    except AttributeError:
        mesh = vps.Mesh()
        mesh.read(vtp_input_path)
        local_domain.fromMesh(mesh)

    depo_model = vps.MultiParticleProcess()
    depo_model.addNeutralParticle(stickingProbability=0.003, label="TMA_flux")
    depo_model.addNeutralParticle(stickingProbability=0.01, label="H2O_flux")

    state = {"coverage": 0.0}
    def rate_fn(fluxes, material):
        tma_flux = fluxes[0]
        theta = state["coverage"]
        d_theta = (tma_flux * theta_sat - theta) / max(tau_tma_s, 1e-6) * 1e-3
        theta = min(max(theta + d_theta, 0.0), theta_sat)
        state["coverage"] = theta
        return gpc_nm * (d_theta / max(theta_sat, 1e-6))

    depo_model.setRateFunction(rate_fn)

    alp_params = vps.AtomicLayerProcessParameters()
    alp_params.numCycles = int(row.num_cycles)
    alp_params.pulseTime = tau_tma_s * 4.0
    alp_params.coverageTimeStep = tau_tma_s / 10.0
    alp_params.purgePulseTime = 0.05

    process = vps.Process(local_domain, depo_model)
    process.setParameters(alp_params)
    process.apply()

    out_vtp = f"{BASE_SAVE_PATH}/topography/run_{i:03d}_{int(row.temperature_C)}C.vtp"
    local_domain.saveSurfaceMesh(out_vtp, True)

    final_mesh = pv.read(out_vtp)
    z_coords = final_mesh.points[:, 2]
    max_z = np.max(z_coords)

    trench_z_coords = z_coords[z_coords > -1.0]
    trench_bottom_z = np.min(trench_z_coords)

    top_thickness = max_z - 42.0
    bottom_thickness = trench_bottom_z - 0.0

    conformality = bottom_thickness / top_thickness if top_thickness > 0 else 1.0
    conformality = float(np.clip(conformality, 0.0, 1.0))

    dataset_records.append({
        "temperature_C": row.temperature_C,
        "pulse_time_s": row.pulse_time_s,
        "num_cycles": row.num_cycles,
        "tma_mole_fraction": row.tma_mole_fraction,
        "gpc_nm_per_cycle": gpc_nm,
        "theta_saturated": theta_sat,
        "total_thickness_nm": gpc_nm * row.num_cycles,
        "conformality_ratio": conformality,
    })
    print(f"  [{i+1:3d}/{N_SAMPLES}] Saved {out_vtp.split('/')[-1]} | Thick: {gpc_nm * row.num_cycles:.2f}nm, Conf: {conformality:.4f}")

pd.DataFrame(dataset_records).to_csv(dataset_csv, index=False)
print(f"\nUnified pipeline complete. Rigorous dataset bound and saved to {dataset_csv}")

import shutil
shutil.copyfile(out_vtp, f"{BASE_SAVE_PATH}/topography/final_deposited_finfet.vtp")


## Surrogate


In [ ]:
import torch
import numpy as np
import pandas as pd
from physicsnemo.models.mlp.fully_connected import FullyConnected
import os

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PhysicsNeMo PINN training running on: {device}")

dataset_csv = f"{BASE_SAVE_PATH}/dataset/ald_high_fidelity_dataset.csv"
if not os.path.exists(dataset_csv):
    raise FileNotFoundError(f"Run Cell 4 first! Unified dataset missing at {dataset_csv}")

dataset_df = pd.read_csv(dataset_csv)
print(f"Loaded {len(dataset_df)} rigorous physical records.")

FEATURE_COLS = ["temperature_C", "pulse_time_s", "num_cycles", "tma_mole_fraction"]
TARGET_COLS = ["total_thickness_nm", "conformality_ratio"]

X = dataset_df[FEATURE_COLS].values.astype(np.float32)
Y = dataset_df[TARGET_COLS].values.astype(np.float32)

X_mean, X_std = X.mean(0), X.std(0) + 1e-8
Y_mean, Y_std = Y.mean(0), Y.std(0) + 1e-8
X_norm = (X - X_mean) / X_std
Y_norm = (Y - Y_mean) / Y_std

n_train = int(0.8 * len(X_norm))
perm = np.random.permutation(len(X_norm))
train_idx, val_idx = perm[:n_train], perm[n_train:]

X_train = torch.tensor(X_norm[train_idx], device=device, requires_grad=True)
Y_train = torch.tensor(Y_norm[train_idx], device=device)
X_val = torch.tensor(X_norm[val_idx], device=device)
Y_val = torch.tensor(Y_norm[val_idx], device=device)

GPC_train = torch.tensor(dataset_df["gpc_nm_per_cycle"].values[train_idx].astype(np.float32), device=device)

surrogate = FullyConnected(
    in_features=len(FEATURE_COLS),
    out_features=len(TARGET_COLS),
    num_layers=4,
    layer_size=64,
).to(device)

model_path = f"{BASE_SAVE_PATH}/models/ald_surrogate_physicsnemo.pt"
optimizer = torch.optim.Adam(surrogate.parameters(), lr=2e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2000)
loss_fn = torch.nn.MSELoss()

N_EPOCHS = 2000
LAMBDA_PHYSICS = 0.5
train_losses, val_losses = [], []

print("\nInitiating Physics-Informed Neural Network (PINN) Training...")

for epoch in range(N_EPOCHS):
    surrogate.train()
    optimizer.zero_grad()

    pred = surrogate(X_train)

    loss_data = loss_fn(pred, Y_train)

    T_pred_norm = pred[:, 0]
    T_pred = T_pred_norm * Y_std[0] + Y_mean[0]

    dT_dX = torch.autograd.grad(
        outputs=T_pred,
        inputs=X_train,
        grad_outputs=torch.ones_like(T_pred),
        create_graph=True,
        retain_graph=True
    )[0]

    dT_dN = dT_dX[:, 2] / X_std[2]

    loss_kinetics = torch.mean((dT_dN - GPC_train) ** 2)

    d2T_dX2 = torch.autograd.grad(
        outputs=dT_dN,
        inputs=X_train,
        grad_outputs=torch.ones_like(dT_dN),
        create_graph=True,
        retain_graph=True
    )[0]
    d2T_dN2 = d2T_dX2[:, 2] / X_std[2]
    loss_linear = torch.mean(d2T_dN2 ** 2)

    X_zero = X_train.clone().detach()
    X_zero[:, 2] = (0.0 - X_mean[2]) / X_std[2]
    pred_zero = surrogate(X_zero)
    T_zero = pred_zero[:, 0] * Y_std[0] + Y_mean[0]
    loss_origin = torch.mean(T_zero ** 2)

    loss_physics = loss_kinetics + loss_linear + loss_origin

    loss = loss_data + (LAMBDA_PHYSICS * loss_physics)

    loss.backward()
    optimizer.step()
    scheduler.step()

    surrogate.eval()
    with torch.no_grad():
        val_pred = surrogate(X_val)
        val_loss = loss_fn(val_pred, Y_val)

    train_losses.append(loss.item())
    val_losses.append(val_loss.item())
    if (epoch + 1) % 200 == 0:
        print(f"  Epoch {epoch+1:4d}/{N_EPOCHS}  Total Loss: {loss.item():.6f}  "
              f"(Data: {loss_data.item():.4f}, Phys: {loss_physics.item():.4f})  "
              f"Val Loss: {val_loss.item():.6f}")

torch.save({
    "model_state_dict": surrogate.state_dict(),
    "X_mean": X_mean, "X_std": X_std,
    "Y_mean": Y_mean, "Y_std": Y_std,
    "feature_cols": FEATURE_COLS, "target_cols": TARGET_COLS,
}, model_path)
print(f"\nSaved trained PhysicsNeMo PINN surrogate to: {model_path}")

loss_curve_path = f"{BASE_SAVE_PATH}/figures/surrogate_training_curve.png"
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4.5))
plt.plot(train_losses, label="Total Train Loss (Data + Physics)")
plt.plot(val_losses, label="Validation Loss")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("PhysicsNeMo PINN Surrogate Training Convergence")
plt.legend()
plt.tight_layout()
plt.savefig(loss_curve_path, dpi=150)
plt.close()
print(f"Saved training convergence curve to: {loss_curve_path}")


## Held-out check


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()

N_TEST = 12
test_sampler = qmc.LatinHypercube(d=4, seed=1234)
test_unit = test_sampler.random(n=N_TEST)
test_scaled = qmc.scale(test_unit, bounds_lo, bounds_hi)
test_df = pd.DataFrame(test_scaled, columns=list(PARAM_BOUNDS.keys()))
test_df["num_cycles"] = test_df["num_cycles"].round().astype(int)

ground_truth_records = []
ground_truth_times = []
for i, row in test_df.iterrows():
    t0 = time.perf_counter()
    gpc_nm, theta_sat = cantera_gpc_at_conditions(
        row.temperature_C, row.pulse_time_s, row.tma_mole_fraction)
    conformality = viennaps_conformality_at_conditions(
        gpc_nm, int(row.num_cycles), theta_sat)
    t1 = time.perf_counter()
    ground_truth_times.append(t1 - t0)
    ground_truth_records.append({
        "total_thickness_nm": gpc_nm * row.num_cycles,
        "conformality_ratio": conformality,
    })
    print(f"  [ground truth {i+1:2d}/{N_TEST}] "
          f"wall time={t1 - t0:.4f}s")

ground_truth_df = pd.DataFrame(ground_truth_records)

checkpoint = torch.load(model_path, map_location=device, weights_only=False)
surrogate.load_state_dict(checkpoint["model_state_dict"])
surrogate.eval()

X_test = test_df[FEATURE_COLS].values.astype(np.float32)
X_test_norm = (X_test - X_mean) / X_std

t0 = time.perf_counter()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_norm, device=device)
    pred_norm = surrogate(X_test_t).cpu().numpy()
t1 = time.perf_counter()
surrogate_total_time = t1 - t0
surrogate_per_sample_time = surrogate_total_time / N_TEST

pred = pred_norm * Y_std + Y_mean
surrogate_df = pd.DataFrame(pred, columns=TARGET_COLS)

comparison_df = pd.concat([
    test_df.reset_index(drop=True),
    ground_truth_df.add_prefix("truth_"),
    surrogate_df.add_prefix("pred_"),
], axis=1)
comparison_df["thickness_rel_error_pct"] = (
    (comparison_df.pred_total_thickness_nm - comparison_df.truth_total_thickness_nm).abs()
    / comparison_df.truth_total_thickness_nm.clip(lower=1e-6) * 100.0
)
comparison_df["conformality_abs_error"] = (
    comparison_df.pred_conformality_ratio - comparison_df.truth_conformality_ratio
).abs()

comparison_csv = f"{BASE_SAVE_PATH}/dataset/surrogate_vs_groundtruth.csv"
comparison_df.to_csv(comparison_csv, index=False)

mean_thickness_error_pct = comparison_df.thickness_rel_error_pct.mean()
mean_conformality_error = comparison_df.conformality_abs_error.mean()
total_ground_truth_time = sum(ground_truth_times)
speedup_factor = total_ground_truth_time / max(surrogate_total_time, 1e-9)

print("\n" + "=" * 70)
print("SURROGATE BENCHMARK SUMMARY")
print("=" * 70)
print(f"  Test set size                      : {N_TEST} held-out process points")
print(f"  Mean film-thickness relative error : {mean_thickness_error_pct:.3f} %")
print(f"  Mean conformality absolute error   : {mean_conformality_error:.5f}")
print(f"  Total ground-truth wall time       : {total_ground_truth_time:.4f} s")
print(f"  Total surrogate wall time          : {surrogate_total_time:.6f} s")
print(f"  Per-sample surrogate time          : {surrogate_per_sample_time * 1000:.4f} ms")
print(f"  Speedup factor (surrogate vs. sim) : {speedup_factor:,.1f}x")
print("=" * 70)
print(f"\nFull comparison table saved to: {comparison_csv}")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(comparison_df.truth_total_thickness_nm,
                 comparison_df.pred_total_thickness_nm, c="#2b6cb0")
lims = [comparison_df[["truth_total_thickness_nm", "pred_total_thickness_nm"]].min().min(),
        comparison_df[["truth_total_thickness_nm", "pred_total_thickness_nm"]].max().max()]
axes[0].plot(lims, lims, "k--", alpha=0.5, label="Perfect agreement")
axes[0].set_xlabel("Ground-truth thickness (nm)")
axes[0].set_ylabel("Surrogate-predicted thickness (nm)")
axes[0].set_title("Film Thickness: Surrogate vs. Ground Truth")
axes[0].legend()

axes[1].bar(["Ground truth\n(Cantera+ViennaPS)", "PhysicsNeMo\nsurrogate"],
            [total_ground_truth_time, surrogate_total_time],
            color=["#c05621", "#2f855a"])
axes[1].set_ylabel("Total wall time for 12 predictions (s)")
axes[1].set_yscale("log")
axes[1].set_title(f"Compute Cost Comparison ({speedup_factor:,.0f}x speedup)")

plt.tight_layout()
benchmark_fig_path = f"{BASE_SAVE_PATH}/figures/surrogate_benchmark.png"
plt.savefig(benchmark_fig_path, dpi=150)
plt.close()
print(f"Saved benchmark comparison figure to: {benchmark_fig_path}")


## Figures


In [ ]:
import pyvista as pv
import numpy as np
import os
import zipfile

_ensure_headless_rendering()

SUBSTRATE_COLOR = "#8a94a6"
STI_COLOR = "#4a7fb5"
DEPOSIT_COLOR = "#ff6a00"

initial_mesh = pv.read(f"{BASE_SAVE_PATH}/geometry/post_photolith_finfet.vtp")
final_mesh = pv.read(f"{BASE_SAVE_PATH}/topography/final_deposited_finfet.vtp")

plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=(1600, 800))

plotter.subplot(0, 0)
plotter.add_mesh(initial_mesh, scalars="MaterialIds" if "MaterialIds" in
                  initial_mesh.point_data else None,
                  cmap=[SUBSTRATE_COLOR, STI_COLOR], show_scalar_bar=False)
plotter.add_text("Post-Photolithography\n(Before ALD)", font_size=14)
plotter.camera_position = "iso"

plotter.subplot(0, 1)
if "MaterialIds" in final_mesh.point_data:
    n_materials = len(np.unique(final_mesh.point_data["MaterialIds"]))
    cmap = [SUBSTRATE_COLOR, STI_COLOR, DEPOSIT_COLOR][:max(n_materials, 1)]
    plotter.add_mesh(final_mesh, scalars="MaterialIds", cmap=cmap,
                      show_scalar_bar=False)
else:
    plotter.add_mesh(final_mesh, color=DEPOSIT_COLOR)
plotter.add_text("After 8-Cycle ALD Al2O3\nDeposition", font_size=14)
plotter.camera_position = "iso"

plotter.link_views()
before_after_path = f"{BASE_SAVE_PATH}/figures/before_after_ald.png"
plotter.screenshot(before_after_path)
plotter.close()
print(f"Saved before/after comparison figure to: {before_after_path}")

cycle_meshes = []
for cycle in range(1, 9):
    path = f"{BASE_SAVE_PATH}/topography/topography_step_{cycle:02d}.vtp"
    try:
        cycle_meshes.append((cycle, pv.read(path)))
    except FileNotFoundError:
        continue

if cycle_meshes:
    n_panels = len(cycle_meshes)
    n_cols = 4
    n_rows = int(np.ceil(n_panels / n_cols))
    grid_plotter = pv.Plotter(shape=(n_rows, n_cols), off_screen=True,
                               window_size=(1800, 450 * n_rows))
    for idx, (cycle, mesh) in enumerate(cycle_meshes):
        r, c = divmod(idx, n_cols)
        grid_plotter.subplot(r, c)
        if "MaterialIds" in mesh.point_data:
            n_mat = len(np.unique(mesh.point_data["MaterialIds"]))
            cmap = [SUBSTRATE_COLOR, STI_COLOR, DEPOSIT_COLOR][:max(n_mat, 1)]
            grid_plotter.add_mesh(mesh, scalars="MaterialIds", cmap=cmap,
                                   show_scalar_bar=False)
        else:
            grid_plotter.add_mesh(mesh, color=DEPOSIT_COLOR)
        grid_plotter.add_text(f"Cycle {cycle}", font_size=10)
        grid_plotter.camera_position = "iso"
    grid_plotter.link_views()
    cycle_grid_path = f"{BASE_SAVE_PATH}/figures/ald_cycle_progression.png"
    grid_plotter.screenshot(cycle_grid_path)
    grid_plotter.close()
    print(f"Saved cycle-by-cycle progression figure to: {cycle_grid_path}")

clip_plotter = pv.Plotter(off_screen=True, window_size=(1200, 900))
clipped = final_mesh.clip(normal="y", origin=final_mesh.center)
if "MaterialIds" in clipped.point_data:
    n_mat = len(np.unique(clipped.point_data["MaterialIds"]))
    cmap = [SUBSTRATE_COLOR, STI_COLOR, DEPOSIT_COLOR][:max(n_mat, 1)]
    clip_plotter.add_mesh(clipped, scalars="MaterialIds", cmap=cmap,
                           show_scalar_bar=True,
                           scalar_bar_args={"title": "Material"})
else:
    clip_plotter.add_mesh(clipped, color=DEPOSIT_COLOR)
clip_plotter.add_text(
    "Cross-Sectional View: Fin-to-Fin Trench\nALD Conformality Inspection",
    font_size=14)
clip_plotter.camera_position = "xz"
cross_section_path = f"{BASE_SAVE_PATH}/figures/trench_conformality_cross_section.png"
clip_plotter.screenshot(cross_section_path)
clip_plotter.close()
print(f"Saved trench conformality cross-section to: {cross_section_path}")

print("\n" + "=" * 70)
print("ALD FinFET DIGITAL TWIN PIPELINE COMPLETE")
print("=" * 70)
print(f"All geometry, chemistry, topography, dataset, model, and figure "
      f"artifacts are saved permanently to {BASE_SAVE_PATH}")

print("Compressing all simulation and surrogate results...")

zip_path = "/content/ald_twin_results.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk(BASE_SAVE_PATH):
        for file in files_list:

            if file.endswith(('.gslides', '.gdoc', '.gsheet', '.gshortcut')):
                continue

            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, BASE_SAVE_PATH)

            try:
                zf.write(file_path, arcname)
            except OSError as e:
                print(f"  [Skipped] Unreadable file ({file}): {e}")

print(f"Archive created at: {zip_path}")
print("Initiating automatic download to your local computer...")

files.download(zip_path)
